# [Práctica 5] Fundamentos de Ciencia de Datos
Alejandra Fernández (afernandezm@inf.udec.cl)

# Contenidos
-  **Codificar variables categoricas**
    - A mano
    - Automatico
- **Clasificacion**
  - Concepto
  - Metricas
  - Modelos
      - SVM
      - Árbol de decisión
      - Random Forest
- **Validación cruzada con un clasificador**
- **Ejercicio**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

#Codificar variables categoricas

Hasta el momento, hemos visto com trabajar con datos númericos, es decir, con variables cuantitativas. Para que los modelos puedan trabajar con variables categoricas, debemos hacer una transformación para poder pasarlas a números.

In [ ]:
#Creemos un dataset de prueba
datos = {
    "Kg" : [2.5, 1.8, 3.2, 1.1, 0.6, 5.0, 4.5, 6.7, 4.3, 2.8],
    "Fruta" : ["Plátano", "Manzana Verde", "Manzana Roja", "Mandarina", "Manzana Roja", "Pera", "Plátano", "Pera", "Manzana Roja", "Manzana Verde"],
    "Importado" : ["No", "Si", "Si", "No", "No", "Si", "Si", "No", "No", "No"],
    "Precio" : [2890, 2500, 3450, 1050, 700, 5400, 5200, 6750, 3100, 1750]
}

tabla = pd.DataFrame(datos)

## A mano

Podemos realizar una transformación "a mano" utilizando un diccionario y la función *replace*

In [ ]:
diccionario = {
    "Plátano" : 1,
    "Manzana Verde" : 2 ,
    "Manzana Roja": 3,
    "Mandarina" : 4,
    "Pera": 5,
}
tabla2 = tabla.replace(diccionario)
tabla2["Importado"].replace({"Si" : 1,"No" : 0},  inplace=True)
tabla2

##Automatico

### Get_dummies
La función get_dummies crea una columna booleana para cada valor categorico. (usa el método de one-hot encoding)

One-hot Encoding es la manera de representar variables categoricas como un vector de 0s y 1s. Solo hay un valor 1, que representa la categoria del elemento (en base a su posición).


In [ ]:
tabla1 = pd.get_dummies(tabla)
tabla1

¿Que pasa si hay un NaN?

In [ ]:
fila_nueva = pd.DataFrame([{"Kg" : 0.9, "Fruta" : float("nan"), "Importado" : "No", "Precio" : 2890}])
tabla3 = pd.concat([tabla, fila_nueva], ignore_index=True)

tabla3

In [ ]:
#Simplemente un nan tendrá todas las columnas con valor 0
pd.get_dummies(tabla3)

Pdemos forzar a que se cree una columna booleana para los nan

In [ ]:
pd.get_dummies(tabla3, dummy_na=True)

### LabelEncoder

Transforma los valores desde 0 hasta el número de clases.

Se aconseja utilizar solo en la variable objetivo ($Y$)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
tabla_aux = tabla.copy()
tabla_aux['Fruta']= le.fit_transform(tabla_aux['Fruta'])
tabla_aux

Puedo obtener el valor original

In [ ]:
le.inverse_transform([1])

In [ ]:
codigos = np.array([tabla_aux.Fruta.unique(), le.inverse_transform(tabla_aux.Fruta.unique())]).T
pd.DataFrame(codigos).sort_values(0)

#Clasificación

##Concepto

El problema de clasificacion consistira en **etiquetar automaticamente** muestras **no observadas** utilizando alguno de estos dos enfoques:
- <u>Modelos Generativos</u>: Determinar la densidad condicionada a cada clase $p(\boldsymbol{x}|C_k)$, donde $C_k$ es la clase y $P(C_k)$ es la probabilidad a priori. Este enfoque utiliza **la probabilidad conjunta** (implicita o explicitamente)
- <u>Modelos Discriminativos</u>:
    - Inferir directamente **el posterior** de la clase $p(C_k|\boldsymbol{x})$ y luego asignar nuevas observaciones a la clase mas apropiada.
    - Aprender una funcion $f(\boldsymbol{x})$, llamada **funcion discriminativa**. la cual mapea las muestras $\boldsymbol{x} \rightarrow C_k$, con $k=0...,K-1$ al espacio de $K$ estiquetas. En este enfoque no utilizamos la probabilidad.  

##Métricas

###¿Que significa TP, TN, FP, FN ?

Pensemos en un modelo que solo nos diga si una imagen es una "Guitarra" o no.

Si yo tengo una entrada X y la etiqueta es "Guitarra" puede suceder:
* El modelo predijo "Guitarra"  -> **True Positive**
* El modelo predijo que no era "Guitarra" -> **False Negative**

Si yo tengo una entrada X' y su etiqueta no es "Guitarra" puede suceder:
* El modelo predijo "Guitarra" -> **False Positive**
* El modelo predijo que no era "Guitarra" -> **True Negative**

En clasificacion solemos utilizar **Accuracy** (exactitud) para evaluar el clasificador. Esta metrica consiste en:

\begin{eqnarray}
\text{Accuracy} = \frac{\text{True Postive} + \textrm{True Negative}}{\text{True Positive} + \text{False Positive} + \text{True Negative} + \text{False Negative}}
\end{eqnarray}

Sin embargo, esta metrica es invariante a conjuntos de datos desbalanceados, para ello podemos utilizar **Balanced Accuracy** (exactitud balanceada):

\begin{eqnarray}
\text{Balanced Accuracy} = \frac{1}{K}\sum_k^K \frac{\text{Casos Positivos}_k}{\text{Casos Totales}_k}
\end{eqnarray}
donde $K$ es el numero de clases

Otra metrica ampliamente utilizada en Clasificacion es el **F1 score** el cual es una media harmonica entre **Precision** y **Recall**
\begin{eqnarray}
    \textrm{F1} & = &\frac{1}{K} \sum_{k=0}^{K-1} 2 \times \frac{\textrm{Precision}_k \times \textrm{Recall}_k}{\textrm{Precision}_k + \textrm{Recall}_k}
    \nonumber
\end{eqnarray}
donde,
\begin{eqnarray}
    \textrm{Recall}_k & = &  \frac{\textrm{True Positives}_k}{\textrm{True Positives}_k + \textrm{False Negatives}_k}
    \nonumber\\
    \textrm{Precision}_k & = &   \frac{\textrm{True Positives}_k}{\textrm{True Positives}_k + \textrm{False Positives}_k}.
    \nonumber
\end{eqnarray}

Intuitivamente, el **Precision** determina cuantos de nuestras predicciones son correctas  mientras que el **Recall** indica el numero de etiquetas verdaderas que fueron correctamente identificadas por el clasificador.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
#Tengo 7 imagenes con etiqueta "Guitarra", de las cuales dije que 4 eran "Guitarra" y el resto no
#Tengo 5 imagenes que no tienen la etiqueta "Guitarra", de las cuales dije que solo 1 era "Guitarra".
test = [1,1,1,1,1,1,1,0,0,0,0,0]
prediccion = [1,1,1,1,0,0,0,0,0,0,0,1]
ConfusionMatrixDisplay.from_predictions(test, prediccion)

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

In [ ]:
accuracy_score(test, prediccion)

In [ ]:
#ACURRACY A MANO
tp =
tn =
fp =
fn =
print((tp+tn)/(tp+tn+fp+fn))

In [ ]:
# ¿Y si tuvieramos 3 clases?
# Clases: "Guitarra" (0), "Bajo" (1), "Bateria" (2)
test_string = ["Bateria","Bateria","Bateria","Bateria","Bajo","Bajo","Bajo","Bajo","Bajo","Bajo","Guitarra","Guitarra","Guitarra","Guitarra","Guitarra"]
pred_string = ["Bajo","Bateria","Bateria","Bateria","Bajo","Bajo","Bateria","Guitarra","Guitarra","Bajo","Guitarra","Guitarra","Guitarra","Guitarra","Guitarra"]
ConfusionMatrixDisplay.from_predictions(test_string, pred_string)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test_string,pred_string))

In [ ]:
# RECALL A MANO

# Guitarra
tp_gui =
fp_gui =
fn_gui =
tn_gui =

# Bajo
tp_baj =
fp_baj =
fn_baj =
tn_baj =

# Bateria
tp_bat =
fp_bat =
fn_bat =
tn_bat =


print("*"*50, "Recall:", "*"*50)

print(f"recall_guitarra: {tp_gui / (tp_gui + fn_gui)}")
print(f"recall_bajo: {tp_baj / (tp_baj + fn_baj)}")
print(f"recall_bateria: {tp_bat / (tp_bat + fn_bat)}")

rec_mean = (
    tp_gui / (tp_gui + fn_gui) +
    tp_baj / (tp_baj + fn_baj) +
    tp_bat / (tp_bat + fn_bat)
) / 3

print(f"recall promedio (a mano): {rec_mean:.3f}")

##Modelos

Vamos a ocupar el dataset de juguete de iris.

En este dataset tenemos los datos:
- longitud del sépalo
- ancho del sépalo
- longitud del pétalo
- ancho del pétalo

Cada fila pertenece a una planta (iris) cuya clase puede ser:
- Setosa
- Versicolour
- Virginica


In [ ]:
from sklearn.datasets import load_iris
iris = load_iris()

In [ ]:
print(iris.DESCR)

In [ ]:
iris_df = pd.DataFrame(data= iris.data, columns= iris.feature_names)
iris_df["species"] = pd.DataFrame(data= iris.target, columns= ['species'])
iris_df.tail(10)

In [ ]:
tipo = pd.DataFrame(data= iris.target_names, columns= ['species'])
tipo

In [ ]:
print("Dimensiones del dataset:", iris_df.shape)
print("\nTipos de datos:")
print(iris_df.dtypes)
print("\nValores nulos:")
print(iris_df.isnull().sum())

In [ ]:
iris_df.describe()

In [ ]:
iris_df["tipo"] = iris_df["species"].replace(tipo.to_dict()["species"])

In [ ]:
iris_df.tipo.hist()
plt.title("Datos por clase")

In [ ]:
import seaborn as sns
sns.countplot(data=iris_df, x="tipo", hue="tipo")

In [ ]:
iris_df.hist(bins=15, edgecolor="black", figsize=(10,8))
plt.suptitle("Histogramas de las características", fontsize=16)
plt.show()

In [ ]:
sns.pairplot(iris_df, hue="tipo", palette="Set2")
plt.suptitle("Relaciones entre características", y=1.02)
plt.show()

In [ ]:
X = iris_df[['sepal length (cm)', 'sepal width (cm)']].values
y = iris_df.species
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=10, shuffle = True)

In [ ]:
for columna in ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']:
  sns.catplot(data=iris_df, x="tipo", y=columna, kind="box", hue='tipo')

### SVM (Support Vector Machine)

<img src = "https://upload.wikimedia.org/wikipedia/commons/b/b5/Svm_separating_hyperplanes_%28SVG%29.svg?utm_source=en.wikipedia.org&utm_campaign=imageinfo&utm_content=original" width="300px">

*Fuente: Wikipedia*

En este algoritmo buscamos optimizar un **hiperplano** que **maximice la distancia** de los **puntos mas cercanos de cada clase**.
\begin{equation} H = \{ x_0, x_2, ..., x_{N-1}\} | \sum_{i=0}^{N-1} a_i x_i = c\end{equation}
Para ilustrar el funcionamiento del algoritmo utilizaremos $N = 2$ dimensiones.

Ademas consideraremos **2 clases** desde el conjunto $X \sim \mathcal{N}(\boldsymbol{\mu}, \sum)$. El problema de optimizacion a resolver consistira en **maximizar el ancho** entre los **puntos mas cercanos al hiperplano** cuya **clase sea distinta**

<img src = "https://upload.wikimedia.org/wikipedia/commons/7/72/SVM_margin.png?utm_source=en.wikipedia.org&utm_campaign=imageinfo&utm_content=original" width="300px">

*Fuente: Wikipedia*

In [ ]:
from sklearn import svm
from sklearn.utils import shuffle
try:
    from mlxtend.plotting import plot_decision_regions
except:
    !pip install mlxtend

#### Entrenamiento

In [ ]:
svm_model = svm.SVC(kernel='linear')
svm_model.fit(X_train, y_train)

#### Predicción

In [ ]:
y_pred = svm_model.predict(X_test)

In [ ]:
plt.figure(figsize=(10,5))
plot_decision_regions(X=X_test,         # Datos
                      y=y_pred,         # Etiquetas
                      clf=svm_model)
plt.title('Limites de decision en SVM')
plt.show()

In [ ]:
aux = [y_test, y_pred]
for num, title in enumerate(["Etiquetas Verdaderas", "Prediccion Support Vector Machine"]):
  cdict = {0: 'lightskyblue', 1: 'orange', 2: 'limegreen'}
  fig, ax = plt.subplots()
  datos = aux[num]
  x_ = X_test[:,0]
  y_ = X_test[:,1]
  for g in np.unique(datos):
    ix = np.where(datos == g)
    ax.scatter(x_[ix],y_[ix], c = cdict[g], label = g, s = 100)
  plt.title(title)
  plt.legend()
plt.show()


#### Evaluación

In [ ]:
# Reporte de clasificación
from sklearn.metrics import classification_report

print(classification_report(y_test,y_pred))

In [ ]:
accuracy_score(y_test,y_pred)

In [ ]:
# Matriz de confusión
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, # Etiquetas correctas
                      y_pred)  # Predicción

print(cm)

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)

#### Otros kernel

Una de las principales limitaciones de los SVM es que solo funcionan con clases **linealmente separables**.

Para resolver esto podemos hacer una **transformacion** del espacio original (tipicamente de mayor dimensionalidad) donde **asumimos** que los puntos seran linealmente separables.

<img  src="https://scikit-learn.org/stable/_images/sphx_glr_plot_iris_svc_001.png">

*Fuente: Scikit-learn*


Para aplicar esta transformación en python, tenemos **kernel** dentro de los argumentos de la función SVM. Para definir el kernel tenemos las siguientes opciones:
- ‘linear‘
- ‘poly’
- ‘rbf’ (por defecto)
- ‘sigmoid’
- ‘precomputed’

In [ ]:
from sklearn.datasets import make_circles
Xnl, ynl = make_circles(100,       # número de muestras.
                        factor=.1, # factor de escalamiento [0,1] entre el círculo interno y externo.
                        noise=.1)  # desviación estandar del ruido gaussiano añadido a los datos.

plt.scatter(Xnl[:, 0], Xnl[:, 1], c=ynl, s=50, cmap='autumn')
plt.show()

In [ ]:
model_svm = svm.SVC(kernel='rbf').fit(Xnl, ynl)

In [ ]:
plt.figure(figsize=(10,5))
plot_decision_regions(X=Xnl,
                      y=ynl,
                      clf=model_svm,
                      legend=1)
plt.show()

In [ ]:
#### Pausa para ocupar todos los atributos del dataset para los siguientes modelos
X = iris_df[['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)','petal width (cm)']].values
y = iris_df.species
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=10, shuffle = True)

In [ ]:
#Entrenar el svm con el dataset completo
svm_model = svm.SVC(kernel='linear')
svm_model.fit(X_train, y_train)
y_pred = svm_model.predict(X_test)

### Árbol de decisión

Otra forma de hacer clasficacion discriminativa es utilizar **arboles de decision**.

Este algoritmo consiste de nodos y aristas. Los **nodos** representan las condiciones $t_i$ de separacion de los datos.

En cada separacion crearemos regiones $R_i$ que contiene un subconjunto de los datos.

Cada region $R_i$ aportara informacion en el proceso discriminativo

El proceso se repite recursivamente hasta que se cumpla algun criterio de detencion.

Las **aristas** determinan el camino hacia los nodos hoja (o Regiones finales de separacion), tipicamente represetan la clasficacion final.

#### Entrenamiento

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn import tree
dt = DecisionTreeClassifier()
dt.fit(X_train,y_train)

#### Mostrar arbolito

In [ ]:
plt.figure(figsize=(10, 6))
tree.plot_tree(dt, filled=True)
plt.show()

#### Predecir

In [ ]:
y_pred_dt = dt.predict(X_test)

In [ ]:
aux = [y_test, y_pred_dt]
for num, title in enumerate(["Etiquetas Verdaderas", "Prediccion Árbol de decisión"]):
  cdict = {0: 'lightskyblue', 1: 'orange', 2: 'limegreen'}
  fig, ax = plt.subplots()
  datos = aux[num]
  x_ = X_test[:,0]
  y_ = X_test[:,1]
  for g in np.unique(datos):
    ix = np.where(datos == g)
    ax.scatter(x_[ix],y_[ix], c = cdict[g], label = g, s = 100)
  plt.title(title)
  plt.legend()
plt.show()

#### Evaluar

In [ ]:
# Reporte de clasificación
from sklearn.metrics import classification_report

print(classification_report(y_test,y_pred_dt))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_dt)

### Random Forest

* Consiste en utilizar N arboles de decision
* Por cada arbol:
 * Seleccionamos $m$ atributos de manera aleatoria (con reemplazo)
 * Entrenamos usando los $m$ atributos  para dividir cada nodo
* $f(x) = \frac{1}{N}\sum_{i=1}^n f_i(x)$, donde $f_i(x)$ es la etiqueta estimada por el $i$-esimo arbol.

#### Entrenar

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

#### Predecir

In [ ]:
y_pred_rf = rf.predict(X_test)

In [ ]:
aux = [y_test, y_pred_rf]
for num, title in enumerate(["Etiquetas Verdaderas", "Prediccion Random Forest"]):
  cdict = {0: 'lightskyblue', 1: 'orange', 2: 'limegreen'}
  fig, ax = plt.subplots()
  datos = aux[num]
  x_ = X_test[:,0]
  y_ = X_test[:,1]
  for g in np.unique(datos):
    ix = np.where(datos == g)
    ax.scatter(x_[ix],y_[ix], c = cdict[g], label = g, s = 100)
  plt.title(title)
  plt.legend()
plt.show()

In [ ]:
rf.feature_importances_

In [ ]:
feature_importances = pd.DataFrame(rf.feature_importances_, index =iris_df.columns[:4],  columns=['importance']).sort_values('importance', ascending=False)
feature_importances

#### Evaluar

In [ ]:
# Reporte de clasificación
from sklearn.metrics import classification_report

print(classification_report(y_test,y_pred_rf))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf)

#Resumen

In [ ]:
metrics = pd.DataFrame()

acc_list = []
bacc_list = []
pre_list = []
reca_list = []
f1_list = []
for pred in [y_pred, y_pred_dt, y_pred_rf]:
    acc_list.append(accuracy_score(y_test, pred))
    bacc_list.append(balanced_accuracy_score(y_test, pred))
    pre_list.append(precision_score(y_test, pred, average='macro'))
    reca_list.append(recall_score(y_test, pred, average='macro'))
    f1_list.append(f1_score(y_test, pred, average='macro'))

nombres=["SVM", "ARBOL DE DECISION", "RANDOM FOREST"]

metrics['Model'] = nombres
metrics['ACC'] = acc_list
metrics['BACC'] = bacc_list
metrics['Precision'] = pre_list
metrics['Recall'] = reca_list
metrics['F1'] = f1_list

metrics.sort_values('F1', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10,5), sharex=True, dpi=100)
for i, pred in enumerate([y_pred, y_pred_dt, y_pred_rf]):
  ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=axes[i])
  axes[i].set_title(nombres[i])
fig.tight_layout()
plt.show()

# Ejercicio

Entrenar distintos random forest (con validación cruzada variar el numero de arboles con los valores 25, 50, 75, 100 y 125) y un modelo SVM lineal para el siguiente conjunto de datos para predecir si pasajero sobrevivió o no

Evaluar ambos modelos (mejor random forest y svm) y comparar matrices de confusión.

En este ejercicio usarás el dataset Titanic para predecir la variable **Survived** (0 = No sobrevivió, 1 = Sí sobrevivió).

 **Variables:**
- Sex: sexo del pasajero/a.
- Age: edad del pasajero/a.
- Survived: si el pasajero/a sobrevivió al naufragio, codificada como 0=no y 1=sí
- pClass: clase a la que pertenecía el pasajero/a: 1, 2 o 3.
- Sibsp: 	# of siblings / spouses aboard the Titanic
- Parch: # of parents / children aboard the Titanic
- Fare: precio pagado por el billete.
- embarked: puerto en el que el pasajero/a embarcó en el Titanic. (C = Cherbourg, Q = Queenstown, S = Southampton)

In [ ]:
import seaborn as sns
titanic = sns.load_dataset('titanic')
titanic['survived'] = titanic['survived'].apply(lambda x: chr(110 + 11*x))

In [ ]:
columnas = ['sex', 'age', 'pclass', 'sibsp', 'parch', 'fare', 'embarked','survived']
titanic = titanic[columnas]
titanic

In [ ]:
#### Tu codigo ####